## 1 — ライブラリの読込み

- 処理: ライブラリの読込み
- 入力: プロジェクトのPython環境
- 出力: 推薦確認と描画の関数
- 次: 実験JSONを指定

- 表はクリックで展開 / 縦横スクロール / CSVは同じ列順の全行・全精度

- 共通評価は新規発見・継続的な嗜好・多様性／発見性・効率性 / 集計は02、定義は[評価仕様](../docs/listening-evaluation.md)


In [ ]:
import json
import tomllib
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from src.experiment_search import inspect_selected, review_selection
from src.notebook_config import ExperimentConfigPicker
from src.notebook_display import show_details, show_table
from src.recommendation_inspection import (
    audio_player,
    feature_comparison,
    inspect_recommendation,
    request_options,
)
from src.search_analysis import workflow_state

## 2 — 実験設定を選択

- 処理: configs内のlistening-v2設定を選択し、読み込むボタンで確定
- 入力: 選択欄のJSON / 初期状態は未選択
- 出力: 読み込んだ実験名・モデル・入力パス / 評価は開始しない
- 次: 読込み済みの表示を確認して次セルへ / ファイル追加後は一覧を更新


In [ ]:
ROOT = Path.cwd() if Path("src").exists() else Path.cwd().parent
config_picker = ExperimentConfigPicker(ROOT / "configs")
display(config_picker.widget)

## 3 — 選択した設定の確認

- 処理: 明示的に読み込んだ設定を後続セルで使用
- 入力: 選択欄の読込み済み設定
- 出力: WORKFLOWと実験名 / 未指定なら案内して停止
- 次: 後続セルを上から順に実行 / 別のNotebookでも同じファイルを選択


In [ ]:
WORKFLOW = config_picker.require_loaded()
EXPERIMENT_CONFIG = WORKFLOW.path
print("実験:", WORKFLOW.name)

## 4 — 選択設定の読込み

- 処理: 選択設定の読込み
- 入力: selection.json・Validation入力
- 出力: モデル・パラメータ・seed・要求一覧 / 未選択ならここで停止
- 次: 確認する要求IDを指定


In [ ]:
config_picker.require_current(globals().get("WORKFLOW"))
if not (WORKFLOW.output / "selection.json").exists():
    raise ValueError(
        "設定は未選択 / 02でValidation結果を確認し、主指標を指定した実験で選択してください"
    )
selected_context = inspect_selected(WORKFLOW)
dataset = selected_context["dataset"]
DATASET_PATH = WORKFLOW.validation_dataset
MODEL = selected_context["model"]
MODEL_PARAMETERS = selected_context["parameters"]
MODEL_SEED = selected_context["seed"]
TOP_K = 20
print(f"{MODEL} / seed {MODEL_SEED} / {dataset.dataset_id}")
if MODEL_PARAMETERS:
    show_details(MODEL_PARAMETERS, "モデル設定")
show_table(
    pd.DataFrame(request_options(dataset))[
        ["task", "user_id", "history_count", "candidate_count", "request_id"]
    ],
    "選択できる要求",
    "03-requests",
)

## 5 — 要求を選んで推薦

- 処理: 要求を選んで推薦
- 入力: REQUEST_ID・選択設定・学習履歴と候補
- 出力: 個別推薦と要求情報 / Test正解は使用しない
- 次: 履歴・順位・根拠を確認


In [ ]:
config_picker.require_current(globals().get("WORKFLOW"))
REQUEST_ID = None  # 上の要求一覧から指定 / Noneは先頭の要求
if REQUEST_ID is None:
    if not dataset.requests:
        raise ValueError("確認できる推薦要求がありません")
    REQUEST_ID = dataset.requests[0].request_id
result = inspect_recommendation(
    dataset,
    model=MODEL,
    request_id=REQUEST_ID,
    parameters=MODEL_PARAMETERS,
    seed=MODEL_SEED,
    features=selected_context.get("features"),
)
print(f"{result.request.task} / user {result.request.user_id} / {result.response.status}")
print(
    f"要求 {TOP_K}曲 / 候補 {len(result.request.candidate_ids)}曲 / 出力 {len(result.response.item_ids)}曲"
)

## 6 — 推薦結果の確認

- 処理: 推薦順位・既聴曲・人気度を図で確認
- 入力: 推薦結果・学習履歴・タグ特徴
- 出力: 順位順の図 / 詳細表は順位 → 曲 → スコア → 既聴 → 再生数 → 根拠
- 次: 音源があれば試聴 / なければ確認記録へ


In [ ]:
config_picker.require_current(globals().get("WORKFLOW"))
history_df = pd.DataFrame(result.history, columns=["item_id", "user_train_events", "train_events"])
recommendations_df = pd.DataFrame(
    result.recommendations,
    columns=["rank", "item_id", "score", "in_history", "train_events", "reason"],
)
if not recommendations_df.empty:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.barh(
        recommendations_df.item_id,
        recommendations_df.train_events,
        color=["#627d98" if seen else "#2a9d8f" for seen in recommendations_df.in_history],
    )
    ax.invert_yaxis()
    ax.set(title="Recommendations", xlabel="Training plays", ylabel="Track")
    plt.tight_layout()
    plt.show()
    print("推薦順位順 / 青灰色は既聴曲、緑は未聴曲")
show_table(recommendations_df, "推薦順位と根拠", "03-recommendations")
show_table(history_df, "ユーザーの履歴", "03-history")
feature_df = pd.DataFrame(feature_comparison(result, selected_context.get("features")))
if not feature_df.empty:
    show_table(feature_df, "特徴比較", "03-features")
else:
    print("音楽的特徴は未取得")
candidates_df = pd.DataFrame(result.candidates).reindex(
    columns=["recommended_rank", "item_id", "in_history", "train_events"]
)
show_table(candidates_df, "候補曲", "03-candidates")

## 7 — ローカル音源の試聴

- 処理: ローカル音源の試聴
- 入力: JSONのinspection.audio_files・AUDITION_ITEM_ID
- 出力: 音源登録状態と手動再生プレイヤー
- 次: 確認内容を記録


In [ ]:
config_picker.require_current(globals().get("WORKFLOW"))
settings = (
    json.loads(EXPERIMENT_CONFIG.read_text())
    if EXPERIMENT_CONFIG.suffix == ".json"
    else tomllib.loads(EXPERIMENT_CONFIG.read_text())
)
AUDIO_FILES = {
    k: (EXPERIMENT_CONFIG.parent / Path(v).expanduser()).resolve()
    for k, v in settings.get("inspection", {}).get("audio_files", {}).items()
}
AUDITION_ITEM_ID = None  # Noneは推薦先頭 / 自動再生なし
AUDITION_ITEM_ID = AUDITION_ITEM_ID or (
    result.response.item_ids[0] if result.response.item_ids else None
)
if AUDITION_ITEM_ID is None:
    print("推薦曲なし / 試聴する履歴曲のIDを指定できます")
else:
    player = audio_player(AUDITION_ITEM_ID, AUDIO_FILES)
    print(AUDITION_ITEM_ID, ":", "再生可能" if player is not None else "音源未登録")
    if player is not None:
        display(player)

## 8 — 確認内容の保存

- 処理: 確認内容の保存
- 入力: REVIEW_ACTION = None / "accept" / "reject"・REVIEW_NOTE
- 出力: 明示時のみ確認artifactを保存
- 次: accept後は02でfreeze / rejectならValidationを見直す


In [ ]:
config_picker.require_current(globals().get("WORKFLOW"))
REVIEW_ACTION = None
REVIEW_NOTE = ""
if REVIEW_ACTION is not None:
    if selected_context is None or REVIEW_ACTION not in {"accept", "reject"}:
        raise ValueError("02の選択artifactとaccept/rejectの明示が必要です")
    review = review_selection(WORKFLOW, accepted=REVIEW_ACTION == "accept", note=REVIEW_NOTE)
    print("確認内容を保存しました")
print("Next:", workflow_state(WORKFLOW)["next_action"])